# OpenAI API Python 실습 노트북

### 실습 목표

- OpenAI API Key를 발급받고 `.env` 파일로 관리한다.
- Python에서 OpenAI API를 호출하는 기본 구조를 익힌다.
- 텍스트 생성 API 호출 과정을 이해한다.
- VLM을 활용하여 이미지와 텍스트를 함께 분석한다.
- API 응답을 로봇 교육 및 로봇 명령 해석 실습에 연결한다.


# 실습 전 주의사항

### API Key 보안

- API Key는 개인 인증 정보에 해당한다.
- API Key를 코드 셀에 직접 작성하지 않는다.
- API Key가 보이는 화면을 캡처하거나 공유하지 않는다.
- API Key가 포함된 파일을 GitHub, Notion, LMS, 메신저 등에 업로드하지 않는다.
- 본 실습에서는 API Key를 `.env` 파일에 저장하고, Python 코드에서 해당 파일을 불러오는 방식을 사용한다.


# OpenAI API Key 발급받기

### API Key 발급 절차

- OpenAI Platform에 로그인한다.
- Dashboard 또는 API Keys 메뉴로 이동한다.
- `Create new secret key` 버튼을 선택한다.
- 구분하기 쉬운 이름을 입력한다.
- 예시 이름은 `python-class-practice`, `vlm-practice`, `robot-api-test` 등이 될 수 있다.
- 생성된 API Key는 다시 확인하기 어려우므로 안전한 위치에 보관한다.
- 발급받은 API Key는 다음 실습에서 `.env` 파일에 입력한다.


# API Key 저장 방식

### .env 파일 사용

- 본 실습에서는 터미널에 환경 변수를 직접 등록하지 않는다.
- 노트북 파일과 같은 폴더에 `.env` 파일을 만든다.
- `.env` 파일은 프로젝트에서 사용하는 비밀값과 설정값을 따로 저장하는 텍스트 파일이다.
- Python 코드는 `.env` 파일을 읽어 `OPENAI_API_KEY` 값을 사용한다.
- `.env` 파일에는 다음 형식으로 작성한다.

```text
OPENAI_API_KEY=여기에_본인의_API_KEY
```

- 변수 이름은 반드시 `OPENAI_API_KEY`로 작성한다.
- 등호 앞뒤에는 불필요한 공백을 넣지 않는다.


# .env 파일 작성 예시

### Windows, Ubuntu, macOS 공통

- `.env` 파일은 일반 텍스트 파일이다.
- VS Code, 메모장, nano 등의 편집기로 작성할 수 있다.
- 작성 예시는 다음과 같다.

```text
OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxxxxxxxxxx
```

- 실제 API Key 전체를 노출하지 않는다.
- `.env` 파일은 GitHub에 올리지 않는다.
- 실제 프로젝트에서는 `.gitignore` 파일에 `.env`를 추가한다.

```text
.env
```


# 필요한 패키지 설치하기

### openai와 python-dotenv 설치

- `openai` 패키지는 OpenAI API를 Python에서 사용하기 위한 공식 SDK이다.
- `python-dotenv` 패키지는 `.env` 파일의 값을 Python 코드로 불러오기 위한 도구이다.
- 아래 셀을 실행하여 두 패키지를 설치한다.


In [ ]:
%pip install -q openai python-dotenv


# .env 파일 확인하기

### API Key 불러오기 확인

- 아래 셀은 현재 노트북과 같은 폴더에서 `.env` 파일을 읽는다.
- `.env` 파일이 없으면 안내 메시지가 출력된다.
- `.env` 파일에 `OPENAI_API_KEY` 값이 있으면 일부만 가려서 표시한다.
- API Key 전체를 출력하지 않도록 구성되어 있다.


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path(".env")

if not env_path.exists():
    print(".env 파일이 없습니다.")
    print("노트북 파일과 같은 폴더에 .env 파일을 만들고 아래처럼 작성하세요.")
    print("OPENAI_API_KEY=여기에_본인의_API_KEY")
else:
    load_dotenv(dotenv_path=env_path, override=True)

    api_key = os.environ.get("OPENAI_API_KEY")

    if api_key:
        masked = api_key[:7] + "..." + api_key[-4:]
        print("OPENAI_API_KEY를 .env에서 불러왔습니다.")
        print("확인용 일부 표시:", masked)
    else:
        print(".env 파일은 있지만 OPENAI_API_KEY 값이 없습니다.")
        print(".env 파일 안에 아래 형식으로 작성했는지 확인하세요.")
        print("OPENAI_API_KEY=여기에_본인의_API_KEY")


# OpenAI 클라이언트 만들기

### client 객체 생성

- `load_dotenv()`는 `.env` 파일의 값을 현재 Python 실행 환경으로 불러온다.
- `client = OpenAI()`는 OpenAI API 요청을 보내기 위한 클라이언트 객체를 만든다.
- `.env` 파일을 정상적으로 불러오면 API Key를 코드에 직접 작성하지 않아도 된다.
- `MODEL_NAME` 변수에는 실습에서 사용할 모델 이름을 저장한다.


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

client = OpenAI()

MODEL_NAME = "gpt-5.5"

print("OpenAI 클라이언트 준비 완료")
print("사용 모델:", MODEL_NAME)


# 첫 번째 텍스트 응답 생성하기

### 기본 API 호출

- 아래 코드는 OpenAI 모델에게 한 문장 설명을 요청한다.
- 요청 결과는 `response` 변수에 저장된다.
- `response.output_text`를 출력하면 모델이 생성한 답변을 확인할 수 있다.
- 같은 질문을 보내더라도 실행 시점에 따라 표현이 달라질 수 있다.


In [ ]:
response = client.responses.create(
    model=MODEL_NAME,
    input="로봇을 처음 배우는 학생에게 ROS2를 한 문장으로 설명해줘."
)

print(response.output_text)


# 코드 이해하기

### 핵심 코드 구조

- `client.responses.create()`는 모델에게 요청을 보내는 함수이다.
- `model`에는 사용할 모델 이름을 입력한다.
- `input`에는 모델에게 전달할 질문이나 명령을 입력한다.
- `response.output_text`에는 모델이 생성한 텍스트 답변이 들어 있다.
- 텍스트 생성 실습에서는 `input`을 바꾸면서 응답 변화를 확인한다.


# 프롬프트 바꿔보기

### 질문을 구체적으로 작성하기

- 프롬프트는 모델에게 전달하는 지시문이다.
- 좋은 답변을 얻으려면 목적, 대상, 형식, 길이, 조건을 구체적으로 작성한다.
- 아래 셀의 `prompt` 값을 수정하면서 결과 차이를 비교한다.


In [ ]:
prompt = """
고등학생이 이해할 수 있게 API를 설명해줘.

조건:
1. 배달앱 주문 과정에 비유하기
2. 어려운 용어는 피하기
3. 마지막에 한 줄 요약 넣기
"""

response = client.responses.create(
    model=MODEL_NAME,
    input=prompt
)

print(response.output_text)


# 설명 수준 바꾸기

### 대상에 맞는 설명 생성

- 같은 주제라도 대상에 따라 설명 방식이 달라져야 한다.
- 아래 실습은 하나의 주제를 초등학생용, 고등학생용, 개발자용으로 나누어 설명하도록 요청한다.
- 프롬프트에서 대상과 출력 형식을 지정하는 방법을 확인한다.


In [ ]:
topic = "API"

prompt = f"""
{topic}를 세 가지 수준으로 설명해줘.

1. 초등학생용
2. 고등학생용
3. 개발자용

각 설명은 3문장 이내로 작성해줘.
"""

response = client.responses.create(
    model=MODEL_NAME,
    input=prompt
)

print(response.output_text)


# 로봇 명령 해석 실습

### 자연어를 구조화된 명령으로 변환

- 본 실습은 실제 로봇을 바로 움직이는 코드가 아니다.
- 사용자의 자연어 명령을 로봇 제어 프로그램에서 사용하기 쉬운 구조로 정리하는 연습이다.
- 실제 로봇 제어와 연결할 때는 안전 조건, 비상 정지, 속도 제한, 사람 감지 절차를 별도로 구현해야 한다.
- 모델의 응답은 명령 후보로만 사용하고, 최종 실행 여부는 프로그램에서 검증해야 한다.


In [ ]:
user_command = "로봇아, 앞으로 천천히 가다가 장애물이 보이면 멈춰."

prompt = f"""
다음 사용자의 말을 로봇 제어 명령으로 해석해줘.

사용자 말:
{user_command}

출력 형식:
동작:
속도:
정지 조건:
주의사항:
"""

response = client.responses.create(
    model=MODEL_NAME,
    input=prompt
)

print(response.output_text)


# VLM 실습

### VLM 개념

- VLM은 Vision-Language Model의 약어이다.
- VLM은 이미지와 텍스트를 함께 입력받아 분석하는 모델이다.
- 텍스트만 입력하면 일반적인 문답을 수행한다.
- 이미지와 질문을 함께 입력하면 이미지 설명, 물체 분석, 위험 요소 확인 등을 수행할 수 있다.
- 로봇 교육에서는 부품 설명, 작업 공간 점검, 센서 이미지 분석 실습에 활용할 수 있다.


# 실습용 샘플 이미지 만들기

### 샘플 이미지 생성

- 아래 셀은 노트북에 포함된 Base64 데이터를 사용하여 `sample_robot_lab.png` 파일을 만든다.
- 생성되는 이미지는 로봇 실습실을 단순화한 예시 그림이다.
- 이후 VLM 실습에서는 이 이미지를 분석 대상으로 사용한다.


In [ ]:
from pathlib import Path
import base64
from IPython.display import Image, display

sample_image_base64 = """..."""

sample_image_path = Path("sample_robot_lab.png")
sample_image_path.write_bytes(base64.b64decode(sample_image_base64))

print("샘플 이미지 생성 완료:", sample_image_path)
display(Image(filename=str(sample_image_path)))


# 로컬 이미지를 Base64로 변환하기

### 이미지 파일을 API 입력 형태로 변환

- 컴퓨터에 저장된 이미지는 API 입력에 바로 넣을 수 없다.
- 이미지 파일을 Base64 문자열로 변환한 뒤 입력 데이터로 전달한다.
- 본 실습에서는 `sample_robot_lab.png` 파일을 읽고 Base64로 변환한다.
- 변환된 문자열은 `data:image/png;base64,...` 형식으로 모델에 전달된다.


In [ ]:
import base64
from pathlib import Path

image_path = Path("sample_robot_lab.png")

image_bytes = image_path.read_bytes()
base64_image = base64.b64encode(image_bytes).decode("utf-8")

print("이미지 파일:", image_path)
print("Base64 길이:", len(base64_image))
print("앞부분 미리보기:", base64_image[:50] + "...")


# VLM으로 이미지 설명하기

### 이미지와 질문을 함께 입력

- VLM 요청에는 텍스트 질문과 이미지 입력이 함께 포함된다.
- `input_text`에는 모델에게 전달할 질문을 입력한다.
- `input_image`에는 모델이 분석할 이미지를 입력한다.
- 아래 셀은 샘플 이미지를 보고 로봇 실습 관점에서 설명하도록 요청한다.


In [ ]:
response = client.responses.create(
    model=MODEL_NAME,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "이 이미지를 한국어로 쉽게 설명해줘. 로봇 실습 관점에서 중요한 부분을 중심으로 설명해줘."
                },
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{base64_image}"
                }
            ]
        }
    ]
)

print(response.output_text)


# 로봇 부품과 위험 요소 찾기

### 로봇 교육용 이미지 분석

- 단순 이미지 설명보다 분석 기준을 구체적으로 제시하면 더 유용한 결과를 얻을 수 있다.
- 아래 실습에서는 주요 물체, 로봇 부품, 예상 역할, 주행 방해 요소, 안전 주의사항을 구분하여 요청한다.
- VLM의 답변은 이미지 기반 추정 결과이다.
- 실제 부품명, 모델명, 안전 판단은 사람이 다시 확인해야 한다.


In [ ]:
prompt = """
이 이미지는 로봇 교육 실습용 이미지다.

다음 기준으로 한국어로 분석해줘.

1. 보이는 주요 물체
2. 로봇 부품으로 보이는 것
3. 각 부품의 예상 역할
4. 로봇 주행에 방해가 될 수 있는 요소
5. 안전상 주의해야 할 점
"""

response = client.responses.create(
    model=MODEL_NAME,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": f"data:image/png;base64,{base64_image}"}
            ]
        }
    ]
)

print(response.output_text)


# VLM 결과를 JSON 형태로 받기

### 프로그램에서 사용하기 쉬운 출력 형식

- 로봇 프로그램과 연결하려면 긴 문장보다 정해진 형식의 데이터가 유리하다.
- JSON 형식은 Python에서 다시 읽고 처리하기 쉽다.
- 아래 실습에서는 이미지 분석 결과를 JSON 형식으로만 출력하도록 요청한다.
- 모델이 형식을 어기면 프롬프트를 더 명확하게 수정해야 한다.


In [ ]:
import json

prompt = """
이미지를 분석해서 반드시 JSON 형식으로만 답해줘.
마크다운 코드블록은 쓰지 마.

형식:
{
  "summary": "이미지 전체 요약",
  "objects": ["보이는 물체 목록"],
  "robot_parts": ["로봇 부품으로 보이는 것"],
  "safety_risks": ["위험 요소"],
  "recommended_action": "실습 전 추천 행동"
}
"""

response = client.responses.create(
    model=MODEL_NAME,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": f"data:image/png;base64,{base64_image}"}
            ]
        }
    ]
)

raw_text = response.output_text
print(raw_text)

print("\n--- JSON 파싱 시도 ---")
try:
    data = json.loads(raw_text)
    print("JSON 파싱 성공")
    print("위험 요소:", data.get("safety_risks"))
except json.JSONDecodeError:
    print("JSON 파싱 실패")
    print("모델이 JSON 이외의 문장을 섞었을 수 있습니다. 프롬프트를 더 강하게 수정해보세요.")


# 내가 준비한 이미지로 바꿔보기

### 직접 촬영한 이미지 분석

- 직접 촬영한 로봇, 센서, 교실, 실습 공간 이미지를 사용할 수 있다.
- 이미지 파일은 노트북 파일과 같은 폴더에 넣는다.
- 예시 코드에서는 파일 이름을 `my_robot.jpg`로 가정한다.
- 다른 파일명을 사용할 경우 `your_image_path` 값을 수정한다.


In [ ]:
from pathlib import Path
import base64
from IPython.display import Image, display

your_image_path = Path("my_robot.jpg")

if not your_image_path.exists():
    print("my_robot.jpg 파일이 없습니다.")
    print("직접 분석할 이미지를 노트북과 같은 폴더에 넣고 파일명을 수정하세요.")
else:
    display(Image(filename=str(your_image_path)))

    your_base64_image = base64.b64encode(your_image_path.read_bytes()).decode("utf-8")

    response = client.responses.create(
        model=MODEL_NAME,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": "이 사진에서 로봇 부품, 센서, 위험 요소를 구분해서 설명해줘."},
                    {"type": "input_image", "image_url": f"data:image/jpeg;base64,{your_base64_image}"}
                ]
            }
        ]
    )

    print(response.output_text)


# 이미지 URL 방식으로 분석하기

### 인터넷 이미지 URL 사용

- 로컬 이미지 대신 인터넷에 공개된 이미지 URL을 사용할 수 있다.
- 웹페이지 주소가 아니라 이미지 파일 자체의 주소를 사용해야 한다.
- 일반적으로 `.jpg`, `.png`, `.webp` 등으로 끝나는 주소가 적절하다.
- 접근 권한이 필요한 이미지는 모델이 불러오지 못할 수 있다.


In [ ]:
image_url = "이미지_URL을_여기에_입력"

if image_url == "이미지_URL을_여기에_입력":
    print("먼저 image_url 변수에 분석할 이미지 URL을 입력하세요.")
else:
    response = client.responses.create(
        model=MODEL_NAME,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": "이 이미지에 무엇이 보이는지 한국어로 설명해줘."},
                    {"type": "input_image", "image_url": image_url}
                ]
            }
        ]
    )

    print(response.output_text)


# 오류 해결

### ModuleNotFoundError

- `No module named 'openai'` 오류는 OpenAI SDK가 설치되지 않았을 때 발생한다.
- 아래 명령을 다시 실행한다.

```python
%pip install openai python-dotenv
```

- 설치 후에도 오류가 계속되면 현재 노트북 커널이 올바른 Python 환경을 사용하는지 확인한다.


# 오류 해결

### API Key 관련 오류

- 인증 오류는 API Key가 없거나 잘못 작성된 경우에 자주 발생한다.
- OpenAI Platform에서 API Key를 정상 발급받았는지 확인한다.
- 노트북 파일과 같은 폴더에 `.env` 파일이 있는지 확인한다.
- `.env` 파일 안의 변수 이름이 `OPENAI_API_KEY`인지 확인한다.
- `.env` 파일을 수정한 뒤 `.env 파일 확인하기` 셀을 다시 실행한다.
- API Key를 코드 셀에 직접 작성하지 않는다.
- `.env` 파일을 GitHub에 업로드하지 않는다.
